# Transaction Anomaly Detection — Consolidated Analysis Notebook

**Group 1 — AI Engineering Assignment**

This single notebook consolidates the five required analysis stages into one place for a quick, narrative walkthrough of the project:

1. **Data Understanding** — schema, dtypes, missing values, duplicates, cardinality
2. **Exploratory Data Analysis** — 11+ visualizations with business interpretation
3. **Preprocessing & Feature Engineering** — cleaning + 14 causal behavioral features
4. **Model Development** — Isolation Forest, LOF, One-Class SVM, Autoencoder (no fraud labels used in training)
5. **Model Evaluation, Comparison & Explainability** — ROC-AUC/AP/precision@k against the held-out fraud label (evaluation only), plus permutation importance and per-transaction explanations

All logic is imported directly from `src/`, so this notebook can never drift out of sync with the production pipeline used by `scripts/*.py` and the Streamlit dashboard.

> **Data note:** this notebook first tries to load the real dataset from `data/raw/` (see `configs/config.yaml`). If it isn't there, it automatically falls back to a small, clearly-labeled **synthetic demo dataset** so every cell still runs — the printed output tells you which mode you're in. Re-run against the real Kaggle dataset before using any numbers from this notebook in the technical report.

*(Per the assignment request, all logic below is intentionally kept in a single, thoroughly-commented code cell rather than split across many cells — see the `# ===` section banners for structure.)*

In [1]:
# =============================================================================
# Transaction Anomaly Detection — Consolidated Analysis Notebook
# Covers: Data Understanding -> EDA -> Preprocessing -> Feature Engineering
#         -> Model Development -> Model Evaluation & Comparison -> Explainability
# =============================================================================
# NOTE ON DATA: this cell first tries to load the REAL raw dataset from
# data/raw/ (per configs/config.yaml). If it isn't present (e.g. you haven't
# downloaded the Kaggle dataset yet), it falls back to a small, CLEARLY
# LABELED synthetic demo dataset with the same schema, purely so every cell
# below still runs end-to-end for a first look at the pipeline mechanics.
# Synthetic-mode results are illustrative only — NOT the project's reported
# evaluation numbers. Run this notebook against the real dataset (see
# README.md) before using any numbers here in the technical report.
# =============================================================================

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config import get_config
from src.data.loader import load_raw_data
from src.data.validator import DataValidator
from src.data.preprocessing import TransactionCleaner, FeatureEncoder
from src.features.feature_engineering import FeatureEngineer
from src.models.train_models import train_all_models, active_model_names
from src.evaluation.evaluate import evaluate_all_models
from src.evaluation.compare_models import build_comparison_table, recommend_model
from src.explainability.explain import AnomalyExplainer
from src.visualization import plots as viz
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)
cfg = get_config()
np.random.seed(cfg.project.get("random_seed", 42))

USING_SYNTHETIC_DEMO_DATA = False
try:
    raw_df = load_raw_data(cfg, sample_frac=cfg.data_loading.get("sample_frac", 0.05))
    print(f"Loaded REAL raw dataset: {len(raw_df):,} rows.")
except FileNotFoundError as e:
    USING_SYNTHETIC_DEMO_DATA = True
    print("!" * 90)
    print("Real dataset not found at", cfg.paths.raw_data)
    print("Falling back to a SMALL SYNTHETIC DEMO dataset so this notebook still runs.")
    print("Download the real dataset from https://www.kaggle.com/datasets/ealtman2019/credit-card-transactions")
    print("and place it at the path above, then re-run this notebook for real results.")
    print("!" * 90)

    def _make_synthetic_demo(n_users=60, txns_per_user=40, seed=42):
        rng = np.random.default_rng(seed)
        merchants = list(range(120))
        rows = []
        for user in range(n_users):
            base_amount = rng.uniform(10, 150)
            day_cursor = 1
            for _ in range(txns_per_user):
                day_cursor += rng.integers(0, 3)
                month = 1 + (day_cursor - 1) // 28
                day = 1 + (day_cursor - 1) % 28
                amount = max(0.5, rng.normal(base_amount, base_amount * 0.3))
                is_fraud = False
                if rng.random() < 0.04:
                    amount *= rng.uniform(15, 50)
                    is_fraud = rng.random() < 0.5
                rows.append({
                    "User": user, "Card": 0, "Year": 2020, "Month": min(month, 12), "Day": day,
                    "Time": f"{rng.integers(0,24):02d}:{rng.integers(0,60):02d}",
                    "Amount": f"${amount:.2f}",
                    "Use Chip": rng.choice(["Chip Transaction", "Swipe Transaction", "Online Transaction"]),
                    "Merchant Name": int(rng.choice(merchants)),
                    "Merchant City": rng.choice(["NYC", "LA", "SF", "ONLINE", "CHI"]),
                    "Merchant State": rng.choice(["NY", "CA", "IL", None]),
                    "Zip": float(rng.integers(10000, 99999)),
                    "MCC": int(rng.choice([5411, 5732, 4829, 5999, 5812, 4900])),
                    "Errors?": rng.choice([None, None, None, "Bad PIN", "Insufficient Balance"]),
                    "Is Fraud?": "Yes" if is_fraud else "No",
                })
        return pd.DataFrame(rows)

    raw_df = _make_synthetic_demo()
    print(f"Generated SYNTHETIC demo dataset: {len(raw_df):,} rows (illustrative only).")

print(raw_df.head())

# =============================================================================
# 1. DATA UNDERSTANDING
# =============================================================================
print("\n" + "=" * 90)
print("1. DATA UNDERSTANDING")
print("=" * 90)
print(f"Dataset shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
print("\nColumn dtypes:\n", raw_df.dtypes)

validator = DataValidator(cfg)
report = validator.profile(raw_df)
print("\nMissing values by column:", report.missing_values)
print("Duplicate rows:", report.n_duplicate_rows)
print("Cardinality (unique values) per key column:", report.cardinality)
print("Inconsistent-value notes:", report.inconsistent_value_notes)
print(
    "\nBusiness problem: identify suspicious card transactions without fraud labels at "
    "training time, since new fraud patterns emerge before they are labeled. "
    "Limitations: the data is synthetic (simulator-generated), so fraud patterns reflect "
    "generative rules rather than real adversarial behavior; Merchant State/Zip are "
    "structurally missing for online transactions; fraud is a rare event (~0.1-2%), so "
    "ranking metrics (ROC-AUC, average precision, precision@k) are used instead of accuracy."
)

# =============================================================================
# 2. PREPROCESSING (cleaning) + FEATURE ENGINEERING
# =============================================================================
print("\n" + "=" * 90)
print("2. PREPROCESSING & FEATURE ENGINEERING")
print("=" * 90)
cleaner = TransactionCleaner(cfg)
clean_df, y = cleaner.clean(raw_df.copy())
print(f"After cleaning: {len(clean_df):,} rows. Fraud-positive rate: {100*y.mean():.3f}%")

fe = FeatureEngineer(cfg)
engineered_df = fe.transform(clean_df)
print(f"Engineered {len(fe.engineered_columns())} behavioral features:")
print(fe.engineered_columns())
print("\nSample of engineered features:")
print(engineered_df[["User", "Amount", *fe.engineered_columns()[:6]]].head())

# =============================================================================
# 3. EXPLORATORY DATA ANALYSIS  (>= 10 visualizations, each with business
#    interpretation printed alongside — see src/visualization/plots.py docstrings)
# =============================================================================
print("\n" + "=" * 90)
print("3. EXPLORATORY DATA ANALYSIS")
print("=" * 90)

eda_df = engineered_df.copy()
eda_df["is_fraud"] = y.values

numeric_corr_cols = [
    "Amount", "customer_avg_amount", "customer_max_amount", "rolling_avg_amount",
    "amount_deviation_from_customer_mean", "amount_zscore_by_merchant",
    "transaction_velocity", "merchant_diversity", "customer_activity_frequency",
]
numeric_corr_cols = [c for c in numeric_corr_cols if c in eda_df.columns]

eda_charts = [
    ("Missing values", lambda: viz.plot_missing_values(raw_df)),
    ("Correlation heatmap", lambda: viz.plot_correlation_heatmap(eda_df, numeric_corr_cols)),
    ("Amount distribution", lambda: viz.plot_amount_distribution(eda_df)),
    ("Amount by channel (boxplot)", lambda: viz.plot_amount_boxplot_by_category(eda_df, "Use Chip")),
    ("Customer transaction frequency", lambda: viz.plot_customer_transaction_frequency(eda_df)),
    ("Merchant distribution", lambda: viz.plot_merchant_distribution(eda_df)),
    ("Hourly transaction pattern", lambda: viz.plot_hourly_transaction_analysis(eda_df)),
    ("Daily transaction trend", lambda: viz.plot_daily_trend(eda_df)),
    ("Weekly transaction trend", lambda: viz.plot_weekly_trend(eda_df)),
    ("Customer spending behavior", lambda: viz.plot_customer_spending_behavior(eda_df)),
    ("Transaction amount by merchant", lambda: viz.plot_transaction_amount_by_merchant(eda_df)),
]

for title, fn in eda_charts:
    print(f"\n--- {title} ---")
    fig = fn()
    plt.show()
    plt.close(fig)

print(f"\nGenerated {len(eda_charts)} visualizations (rubric minimum: 10).")
print("\nDescriptive statistics (numeric columns):")
print(eda_df[numeric_corr_cols].describe().transpose())

# =============================================================================
# 4. ENCODING (fit on train split only) + TRAIN/TEST SPLIT
# =============================================================================
print("\n" + "=" * 90)
print("4. ENCODING & TRAIN/TEST SPLIT")
print("=" * 90)
stratify = y if 0 < y.sum() < len(y) else None
train_df, test_df, y_train, y_test = train_test_split(
    engineered_df, y, test_size=cfg.preprocessing.get("test_size", 0.2),
    random_state=cfg.project.get("random_seed", 42), stratify=stratify,
)

encoder = FeatureEncoder(cfg)
numeric_cols = fe.engineered_columns() + [cfg.data_schema.amount_column]
encoder.fit(train_df, numeric_cols)
feature_cols = encoder.get_feature_columns()
X_train = encoder.transform(train_df)[feature_cols].values
X_test = encoder.transform(test_df)[feature_cols].values

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Features: {len(feature_cols)}")
print("Encoder fit strictly on the training split only (no leakage from test rows).")

# =============================================================================
# 5. MODEL DEVELOPMENT — Isolation Forest, LOF, One-Class SVM, Autoencoder
# =============================================================================
print("\n" + "=" * 90)
print("5. MODEL DEVELOPMENT")
print("=" * 90)
model_names_to_train = [n for n in active_model_names(cfg) if n != "autoencoder"]
try:
    import torch  # noqa: F401
    model_names_to_train = active_model_names(cfg)
    print("PyTorch available — training the Autoencoder as well.")
except ImportError:
    print("PyTorch not available in this environment — training the 3 scikit-learn models only.")
    print("(Install torch per requirements.txt to include the Autoencoder here.)")

models = train_all_models(X_train, cfg, save=False, model_names=model_names_to_train)
for name, model in models.items():
    print(f"  {name:22s} trained in {model.train_time_seconds_:.3f}s | threshold={model.threshold_:.6f}")

# =============================================================================
# 6. MODEL EVALUATION & COMPARISON  (labels used ONLY here, never for fitting)
# =============================================================================
print("\n" + "=" * 90)
print("6. MODEL EVALUATION & COMPARISON")
print("=" * 90)
if USING_SYNTHETIC_DEMO_DATA:
    print("NOTE: metrics below are computed on the SYNTHETIC demo dataset and are "
          "illustrative only — re-run against the real dataset for reportable results.")

results = evaluate_all_models(models, X_test, y_test.values, cfg, save=False)
comparison_table = build_comparison_table(results, cfg)
print(comparison_table[[c for c in ["model","roc_auc","average_precision","precision_at_50","f1","train_time_seconds","prediction_time_seconds"] if c in comparison_table.columns]].to_string(index=False))

recommendation = recommend_model(results, cfg)
print(f"\nRecommended model: {recommendation['recommended_model']}")
print(f"Reason: {recommendation['reason']}")

fig = viz.plot_model_comparison_bar(comparison_table, "roc_auc")
plt.show()
plt.close(fig)

# ROC/PR curves for the top model
top_model_name = comparison_table.iloc[0]["model"]
top_model = models[top_model_name]
top_scores = top_model.score_samples(X_test)
from src.evaluation.metrics import get_roc_curve_data, get_pr_curve_data
roc_data = get_roc_curve_data(y_test.values, top_scores)
pr_data = get_pr_curve_data(y_test.values, top_scores)
if roc_data["fpr"]:
    fig = viz.plot_roc_curve(roc_data["fpr"], roc_data["tpr"], top_model_name, results[top_model_name].get("roc_auc"))
    plt.show(); plt.close(fig)
    fig = viz.plot_precision_recall_curve(pr_data["precision"], pr_data["recall"], top_model_name, results[top_model_name].get("average_precision"))
    plt.show(); plt.close(fig)

fig = viz.plot_anomaly_score_distribution(top_scores, top_model.threshold_, top_model_name)
plt.show(); plt.close(fig)

# =============================================================================
# 7. EXPLAINABILITY
# =============================================================================
print("\n" + "=" * 90)
print("7. EXPLAINABILITY")
print("=" * 90)
explainer = AnomalyExplainer(top_model, feature_cols)
importance_df = explainer.permutation_importance(X_test, n_repeats=3, sample_size=min(3000, len(X_test)))
print(f"\nTop 10 most important features for {top_model_name}:")
print(importance_df.head(10)[["feature", "importance", "description"]].to_string(index=False))

fig = viz.plot_feature_importance(importance_df)
plt.show(); plt.close(fig)

anomaly_positions = np.where(top_model.predict(X_test) == 1)[0]
if len(anomaly_positions) > 0:
    example_idx = anomaly_positions[0]
    explanation = explainer.explain_instance(X_test[example_idx], X_train, top_n=5)
    print(f"\nExample explanation for a flagged transaction (test row {example_idx}):")
    print(" ", explanation["business_explanation"])
    print("\n  Top contributing features:")
    for feat in explanation["top_contributing_features"]:
        print(f"    - {feat['feature']}: value={feat['value']}, z={feat['z_score']:+.2f} ({feat['description']})")
else:
    print("No transactions were flagged as anomalous in this test split.")

if "autoencoder" in models:
    from src.explainability.explain import autoencoder_reconstruction_breakdown
    ae_model = models["autoencoder"]
    ae_scores = ae_model.score_samples(X_test.astype(np.float32))
    fig = viz.plot_reconstruction_error_distribution(ae_scores, ae_model.threshold_)
    plt.show(); plt.close(fig)

    ae_anomaly_positions = np.where(ae_model.predict(X_test.astype(np.float32)) == 1)[0]
    if len(ae_anomaly_positions) > 0:
        idx = ae_anomaly_positions[0]
        breakdown = autoencoder_reconstruction_breakdown(ae_model, X_test[idx], feature_cols, top_n=5)
        print(f"\nAutoencoder reconstruction-error breakdown for test row {idx}:")
        print(" ", breakdown["business_explanation"])

print("\n" + "=" * 90)
print("NOTEBOOK COMPLETE")
print("=" * 90)
print(
    "For the full production pipeline (all 4-5 models, full dataset, persisted artifacts, "
    "and the interactive dashboard), run:\n"
    "  python scripts/run_pipeline.py\n"
    "  streamlit run dashboard/app.py"
)


KeyboardInterrupt: 